# Column Grid Relation Demo

Visual check for furniture geometry, automatically inferred activity regions,
region trend frames, and the explicit column grid. SVG files remain proxy inputs;
the matrix consumes neutral scene and inference records.

In [ ]:
from html import escape
from math import cos, radians, sin
from pathlib import Path

from IPython.display import SVG, display

from features.affordance.models import Rect
from features.column_grid import column_grid_evidence
from features.scene_inference import infer_scene_evidence
from features.scene_loader import load_structured_svg_scene
from features.scene_evaluator import _analysis_scene, _column_grid_relation_rows, _record_axis_descriptor

fixtures = [
    Path("test/SVG_baseline.svg"),
    Path("test/SVG_chaoticsvg.svg"),
    Path("test/SVG_oblique.svg"),
]


def column_grid_matrix(scene):
    """Build the focused column-grid matrix slice used by this visual QA notebook."""
    inference = infer_scene_evidence(scene)
    analysis_scene = _analysis_scene(scene, inferred_regions=inference["inferred_regions"])
    column_grid = column_grid_evidence(analysis_scene)
    # Keep this notebook scoped to column-grid rows while still using the
    # neutral inferred path/network evidence needed by inter-region metrics.
    return {
        "rows": _column_grid_relation_rows(
            column_grid,
            inference["spatial_networks"],
            inference["inferred_region_gap_paths"],
        ),
        "column_grid": column_grid,
        "analysis_scene": analysis_scene,
        "inferred_regions": inference["inferred_regions"],
    }


def column_grid_statuses(matrix):
    """Return column-grid metric rows keyed by metric name for compact visual QA printouts."""
    return {
        row["metric"]: row
        for row in matrix["rows"]
        if row["view_id"] == "column_alignment"
    }


def column_grid_demo_overlay_svg(scene, matrix):
    """Draw only furniture geometry, inferred regions, region trends, and column grid."""
    width, height = scene.get("size", (900.0, 900.0))
    margin_top = 46
    layers = []

    def rect_svg(rect, **attrs):
        """Render one Rect with escaped SVG attributes."""
        attr_text = " ".join(f'{name}="{escape(str(value))}"' for name, value in attrs.items())
        return (
            f'<rect x="{rect.left:.1f}" y="{rect.top + margin_top:.1f}" '
            f'width="{rect.width:.1f}" height="{rect.height:.1f}" {attr_text}/>'
        )

    def polygon_svg(points, **attrs):
        """Render one polygon with escaped SVG attributes."""
        attr_text = " ".join(f'{name}="{escape(str(value))}"' for name, value in attrs.items())
        point_text = " ".join(f"{x:.1f},{y + margin_top:.1f}" for x, y in points)
        return f'<polygon points="{point_text}" {attr_text}/>'

    def shape_svg(record, rect, **attrs):
        """Render original polygon geometry first, with bbox only as fallback."""
        polygon = record.get("polygon")
        points = list(polygon.points) if polygon else list(record.get("points") or [])
        unique_points = {(round(x, 9), round(y, 9)) for x, y in points}
        if len(unique_points) >= 3:
            return polygon_svg(points, **attrs)
        return rect_svg(rect, **attrs)

    def line_svg(x1, y1, x2, y2, **attrs):
        """Render one line while applying the notebook header offset."""
        attr_text = " ".join(f'{name}="{escape(str(value))}"' for name, value in attrs.items())
        return (
            f'<line x1="{x1:.1f}" y1="{y1 + margin_top:.1f}" '
            f'x2="{x2:.1f}" y2="{y2 + margin_top:.1f}" {attr_text}/>'
        )

    def text_svg(x, y, text, **attrs):
        """Render compact labels for source objects and inferred regions."""
        attr_text = " ".join(f'{name}="{escape(str(value))}"' for name, value in attrs.items())
        return f'<text x="{x:.1f}" y="{y + margin_top:.1f}" {attr_text}>{escape(text)}</text>'

    def trend_line(rect, angle, strength, max_length=170):
        """Return a centered trend segment whose length encodes strength."""
        length = max(0.0, min(1.0, strength)) * max_length
        dx = cos(radians(angle)) * length / 2
        dy = sin(radians(angle)) * length / 2
        return rect.cx - dx, rect.cy - dy, rect.cx + dx, rect.cy + dy

    def field_line(rect, angle, strength, max_length=140):
        """Return a centered field segment whose length encodes field strength."""
        length = max(0.0, min(1.0, strength)) * max_length
        dx = cos(radians(angle)) * length / 2
        dy = sin(radians(angle)) * length / 2
        return rect.cx - dx, rect.cy - dy, rect.cx + dx, rect.cy + dy

    legend = [
        (12, "#475569", "furniture"),
        (130, "#2563eb", "column grid"),
        (270, "#059669", "inferred region"),
        (440, "#f97316", "region trend"),
        (580, "#0f766e", "field"),
    ]
    for x, color, label in legend:
        layers.append(f'<rect x="{x}" y="15" width="18" height="10" fill="{color}" opacity="0.65"/>')
        layers.append(f'<text x="{x + 24}" y="24" font-size="13" fill="#334155">{label}</text>')

    for bay in matrix.get("column_grid", {}).get("bays", []):
        rect = bay.get("rect")
        if rect:
            layers.append(shape_svg(bay, rect, fill="#bfdbfe", stroke="#2563eb", **{"stroke-width": 1.4, "opacity": 0.18}))

    for axis in scene.get("column_axes", []):
        points = axis.get("points") or []
        if len(points) >= 2:
            (x1, y1), (x2, y2) = points[0], points[-1]
        else:
            rect = axis["rect"]
            x1, y1, x2, y2 = rect.left, rect.top, rect.right, rect.bottom
        layers.append(line_svg(x1, y1, x2, y2, stroke="#2563eb", **{"stroke-width": 2.3, "opacity": 0.8}))

    for column in scene.get("columns", []):
        layers.append(shape_svg(column, column["rect"], fill="#1d4ed8", stroke="#172554", **{"stroke-width": 1.4, "opacity": 0.55}))

    for obj in scene.get("objects", []):
        rect = obj["rect"]
        layers.append(shape_svg(obj, rect, fill="#64748b", stroke="#334155", **{"stroke-width": 1.0, "opacity": 0.36}))
        layers.append(text_svg(rect.cx, rect.cy + 4, obj["id"], **{"font-size": 9, "text-anchor": "middle", "fill": "#0f172a"}))

    region_by_id = {region["id"]: region for region in matrix.get("analysis_scene", {}).get("regions", [])}

    for region in matrix.get("inferred_regions", []):
        rect = region["rect"]
        # Visual QA must show the inferred activity-region shape that C_local consumes.
        layers.append(shape_svg(region, rect, fill="#10b981", stroke="#047857", **{"stroke-width": 2.2, "stroke-dasharray": "8 5", "opacity": 0.17}))
        axis = _record_axis_descriptor(region)
        metadata = region.get("metadata", {})
        strength = float(metadata.get("axis-strength", 0.0) or 0.0)
        secondary_angle = float(metadata.get("secondary-axis-angle", (axis["angle"] + 90.0) % 180.0) or 0.0)
        # Region trend uses the inferred two-axis frame; segment length encodes trend strength.
        x1, y1, x2, y2 = trend_line(rect, axis["angle"], strength)
        layers.append(line_svg(x1, y1, x2, y2, stroke="#f97316", **{"stroke-width": 2.8, "opacity": 0.9}))
        x1, y1, x2, y2 = trend_line(rect, secondary_angle, strength)
        layers.append(line_svg(x1, y1, x2, y2, stroke="#fb923c", **{"stroke-width": 2.0, "opacity": 0.68}))
        layers.append(f'<circle cx="{rect.cx:.1f}" cy="{rect.cy + margin_top:.1f}" r="4.2" fill="#f97316" opacity="0.9"/>')
        layers.append(text_svg(rect.left, rect.top - 6, region["region"], **{"font-size": 13, "fill": "#047857"}))

    for field_profile in matrix.get("column_grid", {}).get("field_profiles", []):
        region = region_by_id.get(field_profile["region_id"])
        if not region:
            continue
        rect = region["rect"]
        # The field line summarizes continuous structural support; it is not a new grid layer.
        x1, y1, x2, y2 = field_line(rect, field_profile["angle"], field_profile["strength"])
        layers.append(line_svg(x1, y1, x2, y2, stroke="#0f766e", **{"stroke-width": 3.4, "opacity": 0.30 + 0.60 * field_profile["strength"]}))

    return (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width:.0f}" height="{height + margin_top:.0f}" '
        f'viewBox="0 0 {width:.0f} {height + margin_top:.0f}">'
        f'<rect width="100%" height="100%" fill="#ffffff"/>'
        + "".join(layers)
        + "</svg>"
    )


In [ ]:
for path in fixtures:
    scene = load_structured_svg_scene(path)
    matrix = column_grid_matrix(scene)
    column_grid = matrix["column_grid"]
    column_rows = column_grid_statuses(matrix)
    print(
        path.name,
        "columns=", len(scene.get("columns", [])),
        "axes=", len(scene.get("column_axes", [])),
        "bays=", len(column_grid.get("bays", [])),
        "C_local=", column_rows.get("column_grid_C_local_profile", {}).get("status"),
        "inter_region=", column_rows.get("column_grid_inter_region_alignment", {}).get("status"),
        "whole_room=", column_rows.get("column_grid_whole_room_summary", {}).get("status"),
    )
    display(SVG(data=column_grid_demo_overlay_svg(matrix["analysis_scene"], matrix)))


## No-Axis Case

Removing explicit axes should leave the column-grid relation unavailable instead of inferring a grid from physical columns.

In [ ]:
scene = load_structured_svg_scene(Path("test/SVG_baseline.svg"))
scene["column_axes"] = []
matrix = column_grid_matrix(scene)
column_grid = matrix["column_grid"]
column_rows = column_grid_statuses(matrix)
print(
    "no-axis",
    "columns=", len(scene.get("columns", [])),
    "axes=", len(scene.get("column_axes", [])),
    "bays=", len(column_grid.get("bays", [])),
    "C_local=", column_rows.get("column_grid_C_local_profile", {}).get("status"),
    "inter_region=", column_rows.get("column_grid_inter_region_alignment", {}).get("status"),
    "whole_room=", column_rows.get("column_grid_whole_room_summary", {}).get("status"),
    "reason=", column_grid.get("reason"),
)
display(SVG(data=column_grid_demo_overlay_svg(matrix["analysis_scene"], matrix)))
